In [2]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import itertools
import os
import random
import multiprocessing
import sys
import subprocess

BIO_LIP_COLUMNS = [
"PDB ID",
"Receptor chain",
"Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",
"Binding site number code",
"Ligand_ID",
"Ligand_chain",
"Ligand serial number",
"    Binding site residues (with PDB residue numbering)",
"    Binding site residues (with residue re-numbered starting from 1)",
"Catalytic site residues (different sites are separated by ';') (with PDB residue numbering)",
"    Catalytic site residues (different sites are separated by ';') (with residue re-numbered starting from 1)",
"EC number",
"GO terms",
"Binding affinity by manual survey of the original literature. The information in '()' is the PubMed ID",
"Binding affinity provided by the Binding MOAD database. The information in '()' is the ligand information in Binding MOAD",
"Binding affinity provided by the PDBbind-CN database. The information in '()' is the ligand information in PDBbind-CN",
"Binding affinity provided by the BindingDB database",
"UniProt ID",
"PubMed ID",
"Residue sequence number of the ligand (field _atom_site.auth_seq_id in PDBx/mmCIF format)",
"Receptor sequence"]



# now the invalid ligands are stored as strings, written one by one.
INVALID_LIGANDS = ['144', '15P', '1PE', '2F2', '2JC', '3HR', '3SY', '7N5', '7PE', '9JE', 'AAE', 'ABA', 'ACE', 'ACN', 'ACT', 'ACY', 'AZI', 'BAM', 'BCN', 'BCT', 'BDN', 'BEN', 'BME', 'BO3', 'BTB', 'BTC', 'BU1', 'C8E', 'CAD', 'CAQ', 'CBM', 'CCN', 'CIT', 'CL',
'CM', 'CMO', 'CO3', 'CPT', 'CXS', 'D10', 'DEP', 'DIO', 'DMS', 'DN', 'DOD', 'DOX', 'EDO', 'EEE', 'EGL', 'EOH', 'EOX', 'EPE', 'ETF', 'FCY', 'FJO', 'FLC', 'FMT', 'FW5', 'GOL', 'GSH', 'GTT', 'GYF', 'HED', 'IHP', 'IHS', 'IMD', 'IOD', 'IPA', 'IPH',
'LDA', 'MB3', 'MEG', 'MES', 'MLA', 'MLI', 'MOH', 'MPD', 'MRD', 'MSE', 'MYR', 'N', 'NA', 'NH2', 'NH4', 'NHE', 'NO3', 'O4B', 'OHE', 'OLA', 'OLC', 'OMB', 'OME', 'OXA', 'P6G', 'PE3', 'PE4', 'PEG', 'PEO', 'PEP', 'PG0', 'PG4', 'PGE', 'PGR',
'PLM', 'PO4', 'POL', 'POP', 'PVO', 'SAR', 'SCN', 'SEO', 'SEP', 'SIN', 'SO4', 'SPD', 'SPM', 'SR', 'STE', 'STO', 'STU', 'TAR', 'TBU', 'TME', 'TPO', 'TRS', 'UNK', 'UNL', 'UNX', 'UPL', 'URE']


# now with these: SO4, GOL, EDO, PO4, ACT, PEG, DMS, TRS, PGE, PG4, FMT, EPE, MPD, MES, CD, IOD
CRYSTALIZATION_LIGANDS = ['SO4', 'GOL', 'EDO', 'PO4', 'ACT', 'PEG', 'DMS', 'TRS', 'PGE', 'PG4', 'FMT', 'EPE', 'MPD', 'MES', 'CD', 'IOD']

ALL_INVALID_LIGANDS = INVALID_LIGANDS + CRYSTALIZATION_LIGANDS



df = pd.read_csv('/home/iscb/wolfson/hagairavid/LocAlign/BioLiP_nr.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)
relevant_columns = ["PDB ID", "Receptor chain", "Ligand_ID", "Ligand_chain",'Receptor sequence']
df = df[relevant_columns].astype('str')
print(df.shape)
df = df[~df['Ligand_ID'].str.contains('DNA|RNA|PEPTIDE|NONE|nan', case=False, na=False)]
print(df.shape)
df = df[~df['Ligand_ID'].map(lambda x: x in ALL_INVALID_LIGANDS)]

df = df[~df['Receptor chain'].map(lambda x: len(x)>1)]

print(df.shape)


# df = df[:1000:10]
df = df.reset_index(drop=True)

/tmp/ipykernel_697619/3764246858.py:51: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/iscb/wolfson/hagairavid/LocAlign/BioLiP_nr.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)


(82121, 5)
(64854, 5)
(60098, 5)


In [6]:

import shutil

def cluster_sequences(list_sequences, seqid=1.0, coverage=0.8, covmode='0', path2mmseqstmp=None,
                      path2mmseqs=None, threads=8
                      ):

    # Use a Linux-friendly temp directory (defaults to $TMPDIR or a local folder)
    if path2mmseqstmp is None:
        path2mmseqstmp = os.environ.get('TMPDIR', os.path.join(os.getcwd(), 'tmp_mmseqs'))
    os.makedirs(path2mmseqstmp, exist_ok=True)

    # Resolve mmseqs executable
    if path2mmseqs is None:
        mmseqs_exec = shutil.which('mmseqs')
        if mmseqs_exec is None:
            raise FileNotFoundError(
                "mmseqs executable not found in PATH. Install mmseqs2 (e.g., 'conda install -c bioconda mmseqs2') or pass path2mmseqs explicitly."
            )
        path2mmseqs = mmseqs_exec
    else:
        if not os.path.exists(path2mmseqs):
            raise FileNotFoundError(f"mmseqs executable not found at provided path: {path2mmseqs}")

    rng = np.random.randint(0, high=int(1e6))
    tmp_input = os.path.join(path2mmseqstmp, f'tmp_input_file_{rng}.fasta')
    tmp_output = os.path.join(path2mmseqstmp, f'tmp_output_file_{rng}')

    with open(tmp_input, 'w') as f:
        for k, sequence in enumerate(list_sequences):
            f.write(f'>{k}\n')
            f.write(f'{sequence}\n')

    command = ('{mmseqs} easy-cluster {fasta} {result} {tmp} --threads {threads} --min-seq-id %s -c %s --cov-mode %s' % (
        seqid, coverage, covmode)).format(threads=threads, mmseqs=path2mmseqs, fasta=tmp_input, result=tmp_output, tmp=path2mmseqstmp)
    subprocess.run(command.split(' '), check=True)

    with open(tmp_output + '_rep_seq.fasta', 'r') as f:
        representative_indices = [int(x[1:-1]) for x in f.readlines()[::2]]
    cluster_indices = np.zeros(len(list_sequences), dtype=int)
    table_path = tmp_output + '_cluster.tsv'
    table = pd.read_csv(table_path, sep='\t', header=None).to_numpy(dtype=int)
    for i, j in table:
        if i in representative_indices:
            cluster_indices[j] = representative_indices.index(i)
    # Cleanup temporary files if they exist
    for file in [tmp_output + '_rep_seq.fasta', tmp_output + '_all_seqs.fasta', tmp_output + '_cluster.tsv', tmp_input]:
        try:
            if os.path.exists(file):
                os.remove(file)
        except Exception:
            pass
    return np.array(cluster_indices), np.array(representative_indices)


cluster_indices, representative_indices = cluster_sequences(df['Receptor sequence'].to_list(), seqid=0.7,coverage=0.7)



easy-cluster /home/iscb/wolfson/hagairavid/LocAlign/notebooks/tmp_mmseqs/tmp_input_file_959406.fasta /home/iscb/wolfson/hagairavid/LocAlign/notebooks/tmp_mmseqs/tmp_output_file_959406 /home/iscb/wolfson/hagairavid/LocAlign/notebooks/tmp_mmseqs --threads 8 --min-seq-id 0.7 -c 0.7 --cov-mode 0 

MMseqs Version:                     	18.8cc5c
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
Sensitivity                         	4
k-mer length                        	0
Target search mode                  	0
k-score                             	seq:2147483647,prof:2147483647
Alphabet size                       	aa:21,nucl:5
Max sequence length                 	65535
Max results per query               	20
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.7
Coverage mode                       	0


In [9]:
df_nr = df.iloc[representative_indices].reset_index(drop=True)
df_nr = df_nr.drop(columns=['Receptor sequence','Ligand_chain']).rename(columns={'PDB ID':'tar_protein',
                                                                         'Receptor chain':'tar_chain',
                                                                         'Ligand_ID':'tar_ligand',
                                                                         'Ligand_all':'ligand_all'})
df_nr['tar_motif'] = [None for _ in range(len(df_nr))]
df_nr['pocket_residue_ids'] = [None for _ in range(len(df_nr))]  # Will be populated later
df_nr = df_nr[ ['tar_ligand','tar_protein','tar_chain','tar_motif','pocket_residue_ids'] ]
# Don't save yet - will save after populating pocket_residue_ids
print(f"Created df_nr with {len(df_nr)} entries. Ready to extract pocket residues.")

Created df_nr with 23698 entries. Ready to extract pocket residues.


In [10]:
df_nr

,tar_ligand,tar_protein,tar_chain,tar_motif,pocket_residue_ids
0,ASN,11as,A,None,None
1,UPA,11ba,A,None,None
2,ADP,13pk,A,None,None
3,HEM,19hc,A,None,None
4,IPM,1a05,A,None,None
...,...,...,...,...,...
23693,CA,8haw,A,None,None
23694,G2P,8hbf,A,None,None
23695,HEM,8hbh,B,None,None
23696,ATP,8hbw,A,None,None


In [13]:
import sys
sys.path.append('../')

from miners.objects import Protein
from multiprocessing import Pool, cpu_count


def get_pocket_residue_ids(protein_name: str, chain: str, ligand: str, ligand_res_idx: int = 0, distance_thresh: float = 4.0) -> list[int]:
    """
    Extract pocket residue IDs for a given protein-chain-ligand combination.
    Aggregates across all ligand residues (if multiple) and returns a single
    flat, sorted list of unique residue IDs.
    
    Args:
        protein_name: PDB protein name (e.g., '1gkm')
        chain: Chain identifier (e.g., 'A')
        ligand: Ligand identifier (e.g., 'general', 'ADP', etc.)
        ligand_res_idx: Ignored; kept for backward compatibility
        distance_thresh: Pocket cutoff in Å (default: 4.0)
    
    Returns:
        List of residue IDs in the pocket (unique, sorted)
    """
    try:
        protein = Protein(protein_name, chain, ligand, save_models=False)
        ligand_residues = protein.get_ligand_residues()
        pockets = set()
        for i in range(len(ligand_residues)):
            ids = protein.get_pocket_residue_ids(distance_thresh=distance_thresh, ligand_res_idx=i)
            for rid in ids:
                try:
                    pockets.add(int(rid))
                except Exception:
                    pockets.add(rid)
        return sorted(pockets) if pockets else None
    except Exception as e:
        print(f"Error getting pocket for {protein_name} chain {chain} ligand {ligand}: {e}")
        return None


def compute_pocket_for_tuple(args_tuple: tuple[str, str, str], distance_thresh: float = 4.0) -> tuple[list[int] | None, int]:
    """
    Multiprocessing-safe wrapper to compute pocket residue IDs aggregating over
    all ligand residues for a given protein/chain/ligand.
    Returns (sorted_pocket_ids_or_None, ligand_residue_count).
    """
    protein_name, chain, ligand = args_tuple
    try:
        protein = Protein(protein_name, chain, ligand, save_models=False)
        ligand_residues = protein.get_ligand_residues()
        residue_count = len(ligand_residues)
        pockets = set()
        for i in range(residue_count):
            ids = protein.get_pocket_residue_ids(distance_thresh=distance_thresh, ligand_res_idx=i)
            for rid in ids:
                try:
                    pockets.add(int(rid))
                except Exception:
                    pockets.add(rid)
        return (sorted(pockets) if pockets else None), residue_count
    except Exception as e:
        print(f"Error (MP) processing {protein_name} {chain} {ligand}: {e}")
        return None, 0

# Quick test
test_pockets, test_count = compute_pocket_for_tuple(('1gkm', 'A', 'general'))
print(f"Test pockets: {test_pockets} | ligand residues: {test_count}")

Error (MP) processing 1gkm A general: 'A'
Test pockets: None | ligand residues: 0


In [14]:
# Apply pocket residue extraction to all proteins in df_nr (multiprocessing)
print(f"Processing {len(df_nr)} proteins to extract pocket residues (multiprocessing)...")

args_list = [(row['tar_protein'], row['tar_chain'], row['tar_ligand']) for _, row in df_nr.iterrows()]
pocket_residues_list = []
ligand_residue_counts = []

if args_list:
    processes = min(len(args_list), max(1, cpu_count() - 1))
    with Pool(processes=processes) as pool:
        # Preserve order using imap
        for i, (pockets, count) in enumerate(pool.imap(compute_pocket_for_tuple, args_list), 1):
            pocket_residues_list.append(pockets)
            ligand_residue_counts.append(count)
            if i % 50 == 0:
                print(f"Progress: {i}/{len(args_list)}")

# Set columns: tar_motif equals pocket_residue_ids, add ligand residue count
df_nr['pocket_residue_ids'] = pocket_residues_list
df_nr['tar_motif'] = df_nr['pocket_residue_ids']
df_nr['num_ligand_residues'] = ligand_residue_counts

multi_res_cases = (df_nr['num_ligand_residues'] > 1).sum()
print(f"\nCompleted! Added pocket_residue_ids, tar_motif, num_ligand_residues. Multi-residue cases: {multi_res_cases}")
print(f"Sample row: {df_nr.iloc[0]}")

Processing 23698 proteins to extract pocket residues (multiprocessing)...
Progress: 50/23698
Progress: 50/23698
Progress: 100/23698
Progress: 100/23698
Progress: 150/23698
Progress: 150/23698
Progress: 200/23698
Progress: 200/23698
Progress: 250/23698
Progress: 250/23698
Progress: 300/23698
Progress: 350/23698
Progress: 300/23698
Progress: 350/23698
Progress: 400/23698
Progress: 450/23698
Progress: 500/23698
Progress: 550/23698
Progress: 600/23698
Progress: 650/23698
Progress: 700/23698
Progress: 750/23698
Progress: 800/23698
Progress: 400/23698
Progress: 450/23698
Progress: 500/23698
Progress: 550/23698
Progress: 600/23698
Progress: 650/23698
Progress: 700/23698
Progress: 750/23698
Progress: 800/23698
Progress: 850/23698
Progress: 900/23698
Progress: 950/23698
Progress: 1000/23698
Progress: 850/23698
Progress: 900/23698
Progress: 950/23698
Progress: 1000/23698
Progress: 1050/23698
Progress: 1100/23698
Progress: 1150/23698
Progress: 1200/23698
Progress: 1250/23698
Progress: 1300/23698


Failed to parse the downloaded structure 8bd3: The input mmCIF file must begin with a 'data_' directive.


Error (MP) processing 8bd3 m CLA: 0

Progress: 23150/23698
Progress: 23200/23698
Progress: 23150/23698
Progress: 23200/23698
Progress: 23250/23698
Progress: 23300/23698
Progress: 23350/23698
Progress: 23400/23698
Progress: 23450/23698
Progress: 23500/23698
Progress: 23550/23698
Progress: 23600/23698
Progress: 23650/23698
Progress: 23250/23698
Progress: 23300/23698
Progress: 23350/23698
Progress: 23400/23698
Progress: 23450/23698
Progress: 23500/23698
Progress: 23550/23698
Progress: 23600/23698
Progress: 23650/23698

Completed! Added pocket_residue_ids, tar_motif, num_ligand_residues. Multi-residue cases: 5189
Sample row: tar_ligand                                                           ASN
tar_protein                                                         11as
tar_chain                                                              A
tar_motif              [45, 46, 48, 49, 50, 52, 72, 74, 75, 77, 100, ...
pocket_residue_ids     [45, 46, 48, 49, 50, 52, 72, 74, 75, 77, 100, ...
num_li

In [ ]:
# Save the final dataframe with pocket residue IDs

# Ensure tar_motif column entries are lists (or None)
df_nr['tar_motif'] = df_nr['tar_motif'].apply(lambda x: x if (x is None or isinstance(x, list)) else [x])

df_nr = df_nr.drop(columns=['pocket_residue_ids'])

# save full
df_nr.to_csv('../example_inputs/biolip2_nr_database_with_motif.csv', index=False)
print("✅ Saved df_nr with pocket residue IDs and tar_motif lists to 'biolip2_nr_database_with_motif.csv'")

# save mini
df_nr.head(50).to_csv('../example_inputs/biolip2_nr_mini_database_with_motif.csv', index=False)
print("✅ Saved df_nr mini to 'biolip2_nr_mini_database_with_motif.csv'")

print(f"\nDataframe shape: {df_nr.shape}")
print(f"Columns: {list(df_nr.columns)}")
print("tar_motif types:", df_nr['tar_motif'].apply(lambda x: type(x).__name__).value_counts().to_dict())
print(f"\nFirst few rows:")
df_nr.head()

✅ Saved df_nr with pocket residue IDs and tar_motif lists to 'biolip2_nr_database_with_motif.csv'
✅ Saved df_nr mini to 'biolip2_nr_mini_database_with_motif.csv'

Dataframe shape: (23698, 5)
Columns: ['tar_ligand', 'tar_protein', 'tar_chain', 'tar_motif', 'num_ligand_residues']
tar_motif types: {'list': 21748, 'NoneType': 1950}

First few rows:


,tar_ligand,tar_protein,tar_chain,tar_motif,num_ligand_residues
0,ASN,11as,A,"[45, 46, 48, 49, 50, 52, 72, 74, 75, 77, 100, ...",4
1,UPA,11ba,A,"[41, 43, 44, 45, 65, 67, 69, 71, 109, 118, 119...",1
2,ADP,13pk,A,"[217, 218, 219, 223, 241, 242, 245, 294, 314, ...",1
3,HEM,19hc,A,"[8, 10, 11, 14, 16, 17, 18, 29, 30, 31, 32, 33...",9
4,IPM,1a05,A,"[88, 95, 105, 133, 140, 246, 501, 524, 553, 56...",1


In [ ]:
'''
Manual fixes:
1gkm_A => 1gkm_B                                                                                                 | 1/8 [00:11<01:19, 11.33s/it]Failed to download 4ct3 chain A ligand general: 'A'
4ct3_A => 4ct3_E
2h8p_D => delete
'''

In [ ]:
# create mini